In [24]:
# =========================================================
# FEATURE ENGINEERING + MODEL BUILDING
# =========================================================

# -------------------------------
# 1. Import Libraries
# -------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from scipy.sparse import hstack

# -------------------------------
# 2. Load Dataset
# -------------------------------

df = pd.read_csv("zomato_preprocessed.csv")

print("Dataset Shape:", df.shape)

# -------------------------------
# 3. Remove Unlabeled Data
# -------------------------------
# sentiment_encoded has NaN for unknown labels

df = df[df["sentiment_encoded"].notna()]

print("After Removing Unknown Sentiments:", df.shape)

# -------------------------------
# 4. Select Features and Target
# -------------------------------

X_text = df["text_clean"]

# Numerical Features
numerical_features = df[
    ["word_count", "clean_word_count", "char_length"]
]

y = df["sentiment_encoded"]

# -------------------------------
# 5. TF-IDF Feature Engineering
# -------------------------------

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_tfidf = tfidf.fit_transform(X_text)

print("TF-IDF Shape:", X_tfidf.shape)

# -------------------------------
# 6. Scale Numerical Features
# -------------------------------

scaler = StandardScaler()

X_num = scaler.fit_transform(numerical_features)

# -------------------------------
# 7. Combine Text + Numerical Features
# -------------------------------

X = hstack([X_tfidf, X_num])

print("Final Feature Matrix Shape:", X.shape)

# -------------------------------
# 8. Train-Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

# =========================================================
# MODEL BUILDING
# =========================================================

# -------------------------------
# 9. Logistic Regression
# -------------------------------

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred, zero_division=0))

# -------------------------------
# 10. Support Vector Machine
# -------------------------------

svm_model = LinearSVC()

svm_model.fit(X_train, y_train)

svm_pred = svm_model.predict(X_test)

print("\n===== SVM Model =====")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred, zero_division=0))

# -------------------------------
# 11. Naive Bayes
# -------------------------------

nb_model = MultinomialNB()

# Naive Bayes works best with TF-IDF only
X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

nb_model.fit(X_train_nb, y_train_nb)

nb_pred = nb_model.predict(X_test_nb)

print("\n===== Naive Bayes =====")
print("Accuracy:", accuracy_score(y_test_nb, nb_pred))
print(classification_report(y_test_nb, nb_pred, zero_division=0))

# =========================================================
# 12. Predict Custom Sentence
# =========================================================

sample_text = ["Zomato delivery service is excellent and very fast"]

# TF-IDF transform
sample_tfidf = tfidf.transform(sample_text)

# Manual numerical features
sample_num = pd.DataFrame({
    "word_count": [len(sample_text[0].split())],
    "clean_word_count": [len(sample_text[0].split())],
    "char_length": [len(sample_text[0])]
})

sample_num_scaled = scaler.transform(sample_num)

sample_final = hstack([sample_tfidf, sample_num_scaled])

prediction = lr_model.predict(sample_final)

label_map = {
    -1: "Negative",
     0: "Neutral",
     1: "Positive"
}

print("\nPredicted Sentiment:", label_map[int(prediction[0])])

print("\nFeature Engineering and Model Building Completed Successfully")

Dataset Shape: (221, 10)
After Removing Unknown Sentiments: (60, 10)
TF-IDF Shape: (60, 5000)
Final Feature Matrix Shape: (60, 5003)
Training Shape: (48, 5003)
Testing Shape: (12, 5003)

===== Logistic Regression =====
Accuracy: 0.5
              precision    recall  f1-score   support

        -1.0       0.45      1.00      0.62         5
         0.0       1.00      0.25      0.40         4
         1.0       0.00      0.00      0.00         3

    accuracy                           0.50        12
   macro avg       0.48      0.42      0.34        12
weighted avg       0.52      0.50      0.39        12


===== SVM Model =====
Accuracy: 0.5833333333333334
              precision    recall  f1-score   support

        -1.0       0.50      1.00      0.67         5
         0.0       1.00      0.25      0.40         4
         1.0       1.00      0.33      0.50         3

    accuracy                           0.58        12
   macro avg       0.83      0.53      0.52        12
weighted